# Assignment 2

 **This was supoosed to be 2 assignment but is a big assignment So go slow and learn what you are doing have fun**


# Part 1: Data Ingestion

Data Ingestion is the first step in a RAG pipeline. It involves collecting, reading, and loading raw data from various sources (such as PDFs, HTML, or databases) into the system. Here, we read all PDF files in a given directory and parse their content into structured documents containing page content and metadata.


Here we Doing only for pdf but in final project we will do for pdf,csv,xlsx,docx,txt.
if you want you can practise to extract data from one more file format i would love to see you do this.

In [1]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [ ]:
def process_all_pdfs(pdf_directory):

    all_docs =[]
    pdf_files = Path(pdf_directory).glob("**/*.pdf")
    for pdf_file in pdf_file :                  
        loader = PyPDFLoader(str(pdf_file))
        documents = loader.load()  
        for doc in documents :
            doc.metadata["filename"] = pdf_file.name
            doc.metadata["source_path"] = str(pdf_file) 
            if "page" in doc.metadata :
                doc.metadata["page_number"] = doc.metadata["page"]
            all_docs.append(doc)
    return all_docs 
            



In [ ]:
all_pdf_documents = process_all_pdfs("pdf")
all_pdf_documents

# Part 2: Chunking

Chunking is the process of breaking down large documents into smaller, cohesive segments (chunks). Since Large Language Models (LLMs) have a limited context window (input size limit) and retrieve information more accurately from smaller context blocks, we must split large documents. In this assignment, you need to use **RecursiveCharacterTextSplitter** to split loaded documents into smaller paragraphs with proper overlap to maintain context between boundary lines.


In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size ,
        chunk_overlap=chunk_overlap ,
        lenght_function = len ,
        separators=['\n\n', '\n', ' ', '']
    )
    chunks=text_splitter.split_documents(documents)
    return chunks 



In [ ]:
chunks=split_documents(all_pdf_documents)
chunks

# Part 3: Embedding

Embedding is the process of converting text blocks into numerical vector representations. These vectors capture the semantic meaning of the text. Sentences that are semantically similar will be closer to each other in the vector space. We use pre-trained sentence transformer models (like 'all-MiniLM-L6-v2') to convert text chunks and queries into embeddings.

---



from sentence_transformers import SentenceTransformer

Imports the embedding model class.

This library:

downloads pretrained transformer models
converts text → embeddings

Built on top of:

HuggingFace transformers
PyTorch

In [1]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid 
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\rites\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        print(f"loading model : {self.model_name}")
        self.model = SentenceTransformer(self.model_name)
        print("model loaded")
        embed_dim = self.model.get_sentence_embedding_dimension()

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if self.model is None :
            raise ValueError("embedding model is not loaded")
        embeddings = self.model.encode(texts,show_progress_bar=True)
        return embeddings



# Part 4: Vector DB

Vector Database (Vector DB) is a specialized database designed to store and query high-dimensional vector embeddings efficiently. It allows us to persist our document chunks along with their computed embeddings and perform fast search operations. Here, we use ChromaDB, which runs locally and stores documents persistently in a directory.

In [3]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)
        self.client = chromadb.PersistentClient( path=self.persist_directory)
        self.collection = self.client.get_or_create_collectio(name=self.collection_name,metadata={"description": "PDF document embeddings"})

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("No. of docs and embeds must match")
        id = []
        metadatas = []
        docs = []
        embedding_list = []
        for i, doc in enumerate(documents):
            id.append(uuid.uuid4().hex[:8])
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)
            docs.append(doc.page_content)
            embedding_list.append(embeddings[i].tolist())
        self.collection.add(ids=id,embeddings=embedding_list,metadatas=metadatas,documents=docs)


## convert text to embeddings


In [ ]:
texts=[doc.page_content for doc in chunks]
texts

In [ ]:
embedding_manager = EmbeddingManager()
embeddings=embedding_manager.generate_embeddings(texts)
vectorstore=VectorStore()
vectorstore.add_documents(chunks,embeddings)

# Part 5: Query Retrieval

Query Retrieval starts with the user entering a natural language query. We must convert this query into the same embedding space using our embedding manager. This encoded query is then sent to the vector database to retrieve raw results.

---

# Part 6: Similarity Search

Similarity Search is the mathematical calculation (such as Cosine Similarity) used by the vector store to compare the query embedding with document embeddings. It ranks and returns the top_k most similar documents. We can filter results by a minimum similarity score (score_threshold) to keep only relevant context.


In [ ]:
class RAGRetriever:
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        query_embedding =self.embedding_manager.generate_embeddings([query])[0]
        results = self.vector_store.collection.query(query_embedding=[query_embedding.tolist()],n_results=top_k)
        retrieved_docs =[]  # empty list
        l=len(results["ids"][0])
        for i in range(l) :
            distance = results["distances"][0][i]
            sim_score = 1- distance 
            if sim_score >= score_threshold :
                retrieved_docs.append({"id": results["ids"][0][i],"content": results["documents"][0][i],"metadata": results["metadatas"][0][i],"similarity_score": sim_score,"distance": distance,"rank": i + 1
                    })
        return retrieved_docs        


In [ ]:
rag_retriever=RAGRetriever()

In [ ]:
rag_retriever.retrieve(" put something here similar to your data to retrieve")

# Part 7: Prompting and Calling LLM

Prompting and Calling LLM is the synthesis step of RAG. We take the retrieved contexts, format them into a structured prompt alongside the user's original query, and pass them to the Large Language Model (LLM) to generate a grounded, factually accurate response.


In [ ]:
def rag_simple(query,retriever,llm,top_k=3):
    retrieved_docs =retriever.retrieve(query=query , top_k=top_k)
    context ="\n\n".join(doc["content"] for doc in retrieved_docs)
    prompt = f"""use this following context to answer the questions
Context:{context}
Question : {query}
Answer :
"""
    response = llm.invoke(prompt)
    return response.content


In [ ]:
answer=rag_simple("three reasons for forgetting",rag_retriever,llm)  # this will wait for us to import a llm 
print(answer)

# Part 8: Advanced RAG (Optional)

Advanced RAG includes sophisticated pipeline elements such as streaming responses, citation tracking, interaction history (conversational memory), response summarization, and score-based filtering to make the application robust and production-ready.


In [ ]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    retrieved_docs = retriever.retrieve(query=query,top_k=top_k,
                     score_threshold=min_score)

    context = "\n\n".join(doc["content"] for doc in retrieved_docs)
    sources = []
    for doc in retrieved_docs:
        sources.append({"source_file": doc["metadata"].get("filename", "Unknown"),"page": doc["metadata"].get("page_number", "Unknown"),"sim_score": doc["sim_score"],"preview": doc["content"][:150]
                        })
    confidence = 0
    if retrieved_docs:
        confidence = max(doc["sim_score"]for doc in retrieved_docs)

    prompt =f"""
Use the following context to answer the question.
Context:{context}
Question:{query}
Answer:
"""
    response = llm.invoke(prompt)
    result = {"answer": response.content,"sources": sources,"confidence": confidence}

    if return_context:
        result["context"] = context

    return result


In [18]:
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
   def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = [] 

   def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:

    retrieved_docs = self.retriever.retrieve(question,top_k=top_k,score_threshold=min_score)
    context = "\n\n".join(doc["content"] for doc in retrieved_docs)
    sources = []
    for doc in retrieved_docs:
        sources.append({"source_file": doc["metadata"].get("filename", "Unknown"),"page": doc["metadata"].get("page_number", "Unknown"),"sim_score": doc["sim_score"]
             })

    prompt = f"""
Use the following context to answer the question.
Context:{context}
Question:{question}
Answer:
"""
    if stream:
        for i in range(0, len(prompt), 200):
            print(prompt[i:i+200])
            time.sleep(0.05)
    response = self.llm.invoke(prompt)
    answer = response.content
    citations = []
    for source in sources:
        citations.append(f"{source['source_file']} (Page {source['page']})")

    answer_with_citations = (answer +"\n\nSources:\n" +"\n".join(citations))

    summary = None
    if summarize:
        summary_prompt = (f"Summarize the following answer in 2 sentences:\n\n{answer}")

        summary = self.llm.invoke( summary_prompt).content

    record = {"question": question,"answer": answer_with_citations,"sources": sources,"summary": summary}
    self.history.append(record)
    return {"question": question,"answer": answer_with_citations,"sources": sources,"summary": summary,"history": self.history}
